# SCE-Net для задачи совместимости пар одежды (image-only)

Этот ноутбук реализует **Similarity Condition Embedding Network (SCE-Net)** из статьи *Learning Similarity Conditions Without Explicit Supervision* для вашей постановки:
- есть пары `sku1`, `sku2`;
- метка `target in {good, bad}`;
- пути к изображениям `sku1_path`, `sku2_path`.

Мы используем **только визуальную модальность** и берём энкодер признаков из `patrickjohncyh/fashion-clip` (Hugging Face), как вы и попросили.

---

## Как это связано со статьёй

1. **Общий эмбеддинг изображения `V`** (в статье это выход CNN): у нас это выход **FashionCLIP image encoder**.
2. **Similarity condition masks `C_1 ... C_M`** (Раздел 3.1, Eq. 1): обучаемый параметр `[M, D]`, применяем `E_ij = C_j ⊙ V_i`.
3. **Condition weight branch** (Раздел 3.2, Eq. 3): вход `[V_i, V_j]`, MLP + ReLU + Softmax, выход `w` размера `M`.
4. **Агрегация** (Eq. 2): `E_i = w @ O_i`, где `O_i` — masked-векторы `[M, D]`.
5. **Triplet loss** (Eq. 4).
6. **Регуляризация L1/L2** (Eq. 5).

Важно: у вас парные метки good/bad, а triplet требует триплеты. Поэтому ниже строим триплеты из парного датасета.

## 1) Установка и импорты

> Если уже работаете в окружении с `torch/transformers`, этот блок можно пропустить.

In [ ]:
# !pip install -q torch torchvision transformers pandas scikit-learn opencv-python-headless tqdm matplotlib

import os
import random
import time
from dataclasses import dataclass
from typing import Dict

import numpy as np
import pandas as pd
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

from transformers import CLIPModel, CLIPProcessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Для A100 обычно полезно (если input size фиксированный)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

## 2) Конфигурация

- `M` — число similarity conditions (в статье это гиперпараметр; в абляции лучшее около 5).
- `D` берём автоматически из FashionCLIP.

In [ ]:
@dataclass
class Config:
    csv_path: str = 'data/pairs.csv'
    image_root: str = ''

    hf_model_name: str = 'patrickjohncyh/fashion-clip'
    num_conditions: int = 5
    condition_hidden: int = 1024
    dropout: float = 0.1

    batch_size: int = 64
    num_workers: int = 8            # для 28 CPU ядер обычно можно начать с 8-16
    prefetch_factor: int = 4        # сколько батчей готовит каждый worker заранее
    persistent_workers: bool = True # не перезапускать worker-ы каждую эпоху

    lr: float = 1e-4
    weight_decay: float = 1e-5
    epochs: int = 8
    margin: float = 0.2
    lambda_l1: float = 1e-5
    lambda_l2: float = 1e-4

    freeze_backbone: bool = True
    use_amp: bool = True            # mixed precision на A100
    train_size: float = 0.8
    val_size: float = 0.1
    test_size: float = 0.1

cfg = Config()
cfg

## 3) Загрузка и валидация данных

Ожидаемый CSV: `sku1, sku2, target, sku1_path, sku2_path`.
Преобразуем таргет: `good -> 1`, `bad -> 0`.

In [ ]:
def normalize_target(x: str) -> int:
    x = str(x).strip().lower()
    if x in {'good', '1', 'true', 'yes', 'compatible'}:
        return 1
    if x in {'bad', '0', 'false', 'no', 'incompatible'}:
        return 0
    raise ValueError(f'Unknown target value: {x}')


def full_path(p: str, root: str = '') -> str:
    p = str(p)
    if root and not os.path.isabs(p):
        return os.path.join(root, p)
    return p


df = pd.read_csv(cfg.csv_path)
required_cols = {'sku1', 'sku2', 'target', 'sku1_path', 'sku2_path'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df['y'] = df['target'].apply(normalize_target)
df['sku1_path'] = df['sku1_path'].apply(lambda p: full_path(p, cfg.image_root))
df['sku2_path'] = df['sku2_path'].apply(lambda p: full_path(p, cfg.image_root))

mask_exists = df['sku1_path'].map(os.path.exists) & df['sku2_path'].map(os.path.exists)
missing_rows = (~mask_exists).sum()
if missing_rows > 0:
    print(f'WARNING: dropped {missing_rows} rows with missing image files')
df = df[mask_exists].reset_index(drop=True)

print(df.shape)
print(df['y'].value_counts(normalize=True))
df.head()

## 4) Split train/val/test

Стратифицированный split по `y`.

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=(1.0 - cfg.train_size),
    stratify=df['y'],
    random_state=SEED,
)

val_ratio_in_temp = cfg.val_size / (cfg.val_size + cfg.test_size)
val_df, test_df = train_test_split(
    temp_df,
    test_size=(1.0 - val_ratio_in_temp),
    stratify=temp_df['y'],
    random_state=SEED,
)

print('train:', train_df.shape, 'val:', val_df.shape, 'test:', test_df.shape)

## 5) Построение триплетов из парного датасета

Статья обучает SCE-Net через triplet loss. Для anchor `a` берём:
- `p` из good-пар с `a`;
- `n` из bad-пар с `a`.

Так формируем `(a,p,n)` без явных condition labels — в духе статьи.

In [ ]:
def build_affinity_maps(pairs_df: pd.DataFrame):
    pos_map: Dict[str, set] = {}
    neg_map: Dict[str, set] = {}
    path_map: Dict[str, str] = {}

    for row in pairs_df.itertuples(index=False):
        a, b, y = str(row.sku1), str(row.sku2), int(row.y)
        pa, pb = str(row.sku1_path), str(row.sku2_path)
        path_map[a] = pa
        path_map[b] = pb

        if y == 1:
            pos_map.setdefault(a, set()).add(b)
            pos_map.setdefault(b, set()).add(a)
        else:
            neg_map.setdefault(a, set()).add(b)
            neg_map.setdefault(b, set()).add(a)

    return pos_map, neg_map, path_map


def build_triplets(pairs_df: pd.DataFrame, max_triplets_per_anchor: int = 50):
    pos_map, neg_map, path_map = build_affinity_maps(pairs_df)
    triplets = []
    anchors = sorted(set(list(pos_map.keys()) + list(neg_map.keys())))
    rng = random.Random(SEED)

    for a in anchors:
        pos = list(pos_map.get(a, []))
        neg = list(neg_map.get(a, []))
        if len(pos) == 0 or len(neg) == 0:
            continue

        for _ in range(min(max_triplets_per_anchor, len(pos) * len(neg))):
            p = rng.choice(pos)
            n = rng.choice(neg)
            triplets.append((a, p, n, path_map[a], path_map[p], path_map[n]))

    return pd.DataFrame(triplets, columns=[
        'anchor_sku','pos_sku','neg_sku','anchor_path','pos_path','neg_path'
    ])


train_triplets = build_triplets(train_df, max_triplets_per_anchor=40)
val_triplets = build_triplets(val_df, max_triplets_per_anchor=20)

print('train_triplets:', train_triplets.shape)
print('val_triplets:', val_triplets.shape)
train_triplets.head()

## 6) Dataset/DataLoader (подробно)

### Зачем нужен `make_triplet_collate`
В `TripletImageDataset` мы возвращаем **три PIL-изображения** (`anchor`, `positive`, `negative`) на каждый индекс.
`DataLoader` по умолчанию не знает, как эффективно собрать такие объекты в батч для CLIP. Поэтому мы задаём свой `collate_fn`:

1. получает список элементов батча `[(a1,p1,n1), (a2,p2,n2), ...]`;
2. раскладывает его на 3 списка картинок;
3. применяет `CLIPProcessor(..., return_tensors='pt')` отдельно для anchor/positive/negative;
4. возвращает уже готовые `pixel_values` тензоры для модели.

### Зачем в `TripletImageDataset` передаётся `processor`
Сейчас `processor` не используется внутри `__getitem__` (препроцессинг делается в `collate_fn`), но хранится в объекте как:
- удобство для будущего расширения (например, аугментации прямо в dataset),
- явная связь dataset с конкретным image-preprocessor,
- более читаемый API (`TripletImageDataset(..., processor=...)`).


### Критично для вашего кейса: RAM cache изображений
Если `__getitem__` медленный из-за диска, нужно один раз загрузить все уникальные картинки в память
и дальше читать их по ключу `path` из словаря. Ниже это реализовано через `build_image_cache(...)`.


In [ ]:
def build_image_cache(paths, verbose: bool = True):
    """
    Загружает все уникальные изображения в RAM один раз.
    Возвращает dict: {path: np.ndarray(H,W,3) in RGB}.
    """
    unique_paths = sorted(set(map(str, paths)))
    cache = {}

    iterator = tqdm(unique_paths, desc='Caching images to RAM') if verbose else unique_paths
    for p in iterator:
        img_bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        if img_bgr is None:
            raise FileNotFoundError(f'Cannot read image while caching: {p}')
        cache[p] = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    if verbose:
        total_mb = sum(v.nbytes for v in cache.values()) / (1024**2)
        print(f'Cached images: {len(cache)} | approx RAM: {total_mb:.1f} MB')
    return cache


class TripletImageDataset(Dataset):
    def __init__(self, triplet_df: pd.DataFrame, processor: CLIPProcessor, image_cache: dict):
        self.df = triplet_df.reset_index(drop=True)
        self.processor = processor
        self.image_cache = image_cache

    def __len__(self):
        return len(self.df)

    def _load_img(self, p: str):
        # Берём картинку из RAM cache (без чтения с диска)
        p = str(p)
        if p not in self.image_cache:
            raise KeyError(f'Image path not found in RAM cache: {p}')
        return self.image_cache[p]

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return self._load_img(row['anchor_path']), self._load_img(row['pos_path']), self._load_img(row['neg_path'])


def make_triplet_collate(processor: CLIPProcessor):
    """
    Collate для image-only SCE-Net.
    После включения RAM-cache bottleneck обычно смещается в processor/аугментации.
    """
    def collate_fn(batch):
        a_imgs, p_imgs, n_imgs = zip(*batch)
        a_inputs = processor(images=list(a_imgs), return_tensors='pt')
        p_inputs = processor(images=list(p_imgs), return_tensors='pt')
        n_inputs = processor(images=list(n_imgs), return_tensors='pt')
        return a_inputs['pixel_values'], p_inputs['pixel_values'], n_inputs['pixel_values']

    return collate_fn

## 7) Реализация SCE-Net (формулы Eq.1–Eq.3) — подробный разбор

Ниже класс `SCENet` почти строка-в-строку повторяет идею статьи:

- **Eq.1**: `E_ij = C_j ⊙ V_i`
  - `C_j` — j-я mask (обучаемый параметр)
  - `V_i` — общий эмбеддинг изображения
- **Eq.3**: `w = softmax(MLP(concat(V_i, V_j)))`
  - branch вычисляет релевантность каждого условия под конкретную пару
- **Eq.2**: `E_i = w @ O_i`
  - `O_i` — набор masked-эмбеддингов `[M, D]`

Ключевая мысль: сравнение двух айтемов выполняется в **динамически выбранном semantic subspace**.


In [ ]:
class SCENet(nn.Module):
    def __init__(self, clip_model_name: str, num_conditions: int = 5, hidden_dim: int = 1024, dropout: float = 0.1):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(clip_model_name)

        if hasattr(self.clip, 'visual_projection') and self.clip.visual_projection is not None:
            proj = self.clip.visual_projection
            emb_dim = proj.out_features if hasattr(proj, 'out_features') else self.clip.config.projection_dim
        else:
            emb_dim = getattr(self.clip.config, 'projection_dim', None)
            if emb_dim is None:
                raise ValueError('Cannot infer embedding dimension D from model config/projection.')

        self.emb_dim = emb_dim
        self.num_conditions = num_conditions

        # Eq.1: similarity condition masks C in R^{M x D}
        self.condition_masks = nn.Parameter(torch.empty(num_conditions, emb_dim))
        nn.init.xavier_uniform_(self.condition_masks)

        # Eq.3: condition weight branch
        self.weight_branch = nn.Sequential(
            nn.Linear(2 * emb_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_conditions)
        )

    def _extract_image_features(self, pixel_values: torch.Tensor) -> torch.Tensor:
        if hasattr(self.clip, 'get_image_features'):
            out = self.clip.get_image_features(pixel_values=pixel_values)
            if torch.is_tensor(out):
                return out
            if hasattr(out, 'image_embeds') and out.image_embeds is not None:
                return out.image_embeds
            if hasattr(out, 'pooler_output') and out.pooler_output is not None:
                return out.pooler_output
            if hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
                return out.last_hidden_state[:, 0, :]

        out = self.clip(pixel_values=pixel_values, return_dict=True)
        if hasattr(out, 'image_embeds') and out.image_embeds is not None:
            return out.image_embeds
        if hasattr(out, 'pooler_output') and out.pooler_output is not None:
            return out.pooler_output
        if hasattr(out, 'last_hidden_state') and out.last_hidden_state is not None:
            return out.last_hidden_state[:, 0, :]

        raise TypeError(f'Unsupported image output type: {type(out)}')

    def encode_image(self, pixel_values: torch.Tensor) -> torch.Tensor:
        img_feat = self._extract_image_features(pixel_values)
        return F.normalize(img_feat, p=2, dim=-1)

    def compute_w(self, v1: torch.Tensor, v2: torch.Tensor) -> torch.Tensor:
        # Eq.3: w = softmax(MLP(concat(Vi, Vj)))
        logits = self.weight_branch(torch.cat([v1, v2], dim=-1))
        return F.softmax(logits, dim=-1)

    def apply_conditions(self, v: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
        # Eq.1 + Eq.2
        masked = v.unsqueeze(1) * self.condition_masks.unsqueeze(0)
        return torch.einsum('bm,bmd->bd', w, masked)

    def project_pair_from_embeddings(self, v1: torch.Tensor, v2: torch.Tensor):
        # Удобно для оптимизации: если v уже посчитаны, не гоняем encoder ещё раз
        w = self.compute_w(v1, v2)
        e1 = self.apply_conditions(v1, w)
        e2 = self.apply_conditions(v2, w)
        return e1, e2, w

    def forward_pair(self, img1: torch.Tensor, img2: torch.Tensor):
        v1 = self.encode_image(img1)
        v2 = self.encode_image(img2)
        e1, e2, w = self.project_pair_from_embeddings(v1, v2)
        return e1, e2, w, v1, v2

    def forward_triplet(self, a_px: torch.Tensor, p_px: torch.Tensor, n_px: torch.Tensor):
        """
        Оптимизированный forward для триплета:
        anchor кодируется ОДИН раз (а не два, как в двух вызовах forward_pair).
        Это снижает нагрузку на backbone и убирает лишние вычисления.
        """
        v_a = self.encode_image(a_px)
        v_p = self.encode_image(p_px)
        v_n = self.encode_image(n_px)

        # Пара (a,p)
        e_a_pos, e_p, w_ap = self.project_pair_from_embeddings(v_a, v_p)
        # Пара (a,n)
        e_a_neg, e_n, w_an = self.project_pair_from_embeddings(v_a, v_n)

        # Как и раньше, усредняем два anchor-представления
        e_a = 0.5 * (e_a_pos + e_a_neg)
        return e_a, e_p, e_n, w_ap, w_an

## 8) Loss (Eq.4 + Eq.5) — подробный разбор

Функция `sce_loss` реализует objective из статьи:

\[
L = L_{triplet} + \lambda_1 \cdot L_1(C) + \lambda_2 \cdot L_2(E)
\]

Где:
- `d_pos = ||E_a - E_p||_2` — расстояние anchor до positive;
- `d_neg = ||E_a - E_n||_2` — расстояние anchor до negative;
- `triplet = max(0, d_pos - d_neg + margin)` (Eq.4).

Почему именно такие строки:
- `torch.norm(..., p=2, dim=-1)` — это евклидова дистанция по embedding-компонентам;
- нам нужно, чтобы `d_pos < d_neg` хотя бы на `margin`.

Если условие уже выполнено, вклад триплета 0; если нет — модель получает штраф.


In [ ]:
def sce_loss(e_anchor, e_pos, e_neg, condition_masks, margin, lambda_l1, lambda_l2):
    # Евклидовы расстояния (Eq.4)
    # d_pos: насколько anchor близок к positive (хотим МЕНЬШЕ)
    d_pos = torch.norm(e_anchor - e_pos, p=2, dim=-1)
    # d_neg: насколько anchor далёк от negative (хотим БОЛЬШЕ)
    d_neg = torch.norm(e_anchor - e_neg, p=2, dim=-1)

    # Triplet hinge: max(0, d_pos - d_neg + margin)
    # Если d_neg >= d_pos + margin, ошибка по этому примеру = 0
    triplet = F.relu(d_pos - d_neg + margin).mean()

    # L1 на masks C: по статье стимулирует sparsity/disentanglement условий
    l1 = condition_masks.abs().mean()

    # L2 на репрезентации: регуляризуем масштаб embedding-ов
    l2 = (e_anchor.pow(2).mean() + e_pos.pow(2).mean() + e_neg.pow(2).mean()) / 3.0

    # Финальная функция (Eq.5)
    total = triplet + lambda_l1 * l1 + lambda_l2 * l2

    # Возвращаем и loss tensor, и удобные scalar-метрики для логирования
    return total, {'loss': total.item(), 'triplet': triplet.item(), 'l1': l1.item(), 'l2': l2.item()}

## 9) Инициализация обучения

In [ ]:
processor = CLIPProcessor.from_pretrained(cfg.hf_model_name)

# ------------------------------------------------------------
# 1) Единоразовая загрузка всех уникальных изображений в RAM
# ------------------------------------------------------------
all_paths = pd.concat([df['sku1_path'], df['sku2_path']], axis=0).astype(str).tolist()
IMAGE_CACHE = build_image_cache(all_paths, verbose=True)

# ------------------------------------------------------------
# 2) Dataset/DataLoader используют RAM cache, а не диск в __getitem__
# ------------------------------------------------------------
train_ds = TripletImageDataset(train_triplets, processor, image_cache=IMAGE_CACHE)
val_ds = TripletImageDataset(val_triplets, processor, image_cache=IMAGE_CACHE)
collate_fn = make_triplet_collate(processor)

loader_kwargs = dict(
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
if cfg.num_workers > 0:
    loader_kwargs['prefetch_factor'] = cfg.prefetch_factor
    loader_kwargs['persistent_workers'] = cfg.persistent_workers

train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)

model = SCENet(cfg.hf_model_name, cfg.num_conditions, cfg.condition_hidden, cfg.dropout).to(device)

if cfg.freeze_backbone:
    for p in model.clip.parameters():
        p.requires_grad = False

optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg.lr, weight_decay=cfg.weight_decay)
scaler = torch.cuda.amp.GradScaler(enabled=(cfg.use_amp and torch.cuda.is_available()))

print('trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))
print('embedding dim D:', model.emb_dim)

## 10) Train/Val loop (оптимизировано для A100)

Почему у вас могла быть "пилообразная" утилизация GPU и деградация по эпохам:

1. **Двойной encode anchor** в каждом батче (`forward_pair(a,p)` + `forward_pair(a,n)`) → лишние вычисления.
2. **CPU bottleneck на PIL/processor**: декодинг JPEG и preprocess часто не успевают кормить GPU.
3. **Неоптимальные DataLoader-параметры**: мало worker-ов, нет prefetch/persistent workers.
4. **Отсутствие AMP**: на A100 mixed precision обычно заметно ускоряет шаг.
5. **Блокирующий copy CPU→GPU**: без `non_blocking=True` pipeline менее эффективен.

Ниже правки:
- `model.forward_triplet(...)` кодирует anchor один раз;
- `autocast + GradScaler`;
- тайминг data/compute на эпоху, чтобы видеть узкое место;
- proxy ROC-AUC по триплет-парам (good=(a,p), bad=(a,n)).


In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    losses, triplets = [], []
    y_true, y_score = [], []

    data_time = 0.0
    compute_time = 0.0
    t_prev = time.perf_counter()

    for a_px, p_px, n_px in tqdm(loader, leave=False):
        t_after_load = time.perf_counter()
        data_time += (t_after_load - t_prev)

        # Асинхронный перенос в GPU при pin_memory=True
        a_px = a_px.to(device, non_blocking=True)
        p_px = p_px.to(device, non_blocking=True)
        n_px = n_px.to(device, non_blocking=True)

        t0 = time.perf_counter()
        with torch.set_grad_enabled(is_train):
            with torch.cuda.amp.autocast(enabled=(cfg.use_amp and torch.cuda.is_available())):
                e_a, e_p, e_n, _, _ = model.forward_triplet(a_px, p_px, n_px)
                loss, stats = sce_loss(e_a, e_p, e_n, model.condition_masks, cfg.margin, cfg.lambda_l1, cfg.lambda_l2)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        compute_time += (time.perf_counter() - t0)

        losses.append(stats['loss'])
        triplets.append(stats['triplet'])

        ap_score = (-torch.norm(e_a - e_p, p=2, dim=-1)).detach().cpu().numpy()
        an_score = (-torch.norm(e_a - e_n, p=2, dim=-1)).detach().cpu().numpy()
        y_score.extend(ap_score.tolist())
        y_score.extend(an_score.tolist())
        y_true.extend([1] * len(ap_score))
        y_true.extend([0] * len(an_score))

        t_prev = time.perf_counter()

    out = {
        'loss': float(np.mean(losses)) if losses else np.nan,
        'triplet': float(np.mean(triplets)) if triplets else np.nan,
        'data_time_s': float(data_time),
        'compute_time_s': float(compute_time),
    }

    if len(set(y_true)) == 2:
        out['triplet_pair_auc'] = float(roc_auc_score(y_true, y_score))
    else:
        out['triplet_pair_auc'] = np.nan

    return out


best_val = float('inf')
best_path = 'sce_net_best.pt'
history = []

for epoch in range(1, cfg.epochs + 1):
    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, optimizer=None)

    history.append({'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'val_{k}': v for k, v in val_metrics.items()}})

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_metrics['loss']:.4f} | val_loss={val_metrics['loss']:.4f} | "
        f"train_triplet_auc={train_metrics['triplet_pair_auc']:.4f} | "
        f"val_triplet_auc={val_metrics['triplet_pair_auc']:.4f} | "
        f"train(data/compute)={train_metrics['data_time_s']:.1f}/{train_metrics['compute_time_s']:.1f}s"
    )

    if val_metrics['loss'] < best_val:
        best_val = val_metrics['loss']
        torch.save({'model_state': model.state_dict(), 'config': cfg.__dict__}, best_path)
        print('  -> saved best checkpoint:', best_path)

pd.DataFrame(history)

## 11) Pair scoring и метрики

На инференсе для пары считаем `score = -||E_i - E_j||` и оцениваем AUC/PR-AUC.

In [ ]:
@torch.no_grad()
def pair_scores(model: SCENet, pair_df: pd.DataFrame, processor: CLIPProcessor, batch_size: int = 64, image_cache: dict = None):
    model.eval()
    scores = []
    labels = pair_df['y'].astype(int).tolist()

    rows = pair_df[['sku1_path', 'sku2_path']].astype(str).values.tolist()
    for i in tqdm(range(0, len(rows), batch_size)):
        chunk = rows[i:i+batch_size]
        imgs1, imgs2 = [], []
        for a, b in chunk:
            if image_cache is not None:
                if a not in image_cache or b not in image_cache:
                    raise KeyError(f'Pair path not found in cache: {a}, {b}')
                imgs1.append(image_cache[a])
                imgs2.append(image_cache[b])
            else:
                a_bgr = cv2.imread(a, cv2.IMREAD_COLOR)
                b_bgr = cv2.imread(b, cv2.IMREAD_COLOR)
                if a_bgr is None or b_bgr is None:
                    raise FileNotFoundError(f'Cannot read image in pair: {a}, {b}')
                imgs1.append(cv2.cvtColor(a_bgr, cv2.COLOR_BGR2RGB))
                imgs2.append(cv2.cvtColor(b_bgr, cv2.COLOR_BGR2RGB))

        px1 = processor(images=imgs1, return_tensors='pt')['pixel_values'].to(device)
        px2 = processor(images=imgs2, return_tensors='pt')['pixel_values'].to(device)

        e1, e2, _, _, _ = model.forward_pair(px1, px2)
        scores.extend((-torch.norm(e1 - e2, p=2, dim=-1)).cpu().numpy().tolist())

    return np.array(scores), np.array(labels)


ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt['model_state'])

val_scores, val_labels = pair_scores(model, val_df, processor, image_cache=IMAGE_CACHE)
print('VAL ROC-AUC=', roc_auc_score(val_labels, val_scores), 'PR-AUC=', average_precision_score(val_labels, val_scores))

test_scores, test_labels = pair_scores(model, test_df, processor, image_cache=IMAGE_CACHE)
print('TEST ROC-AUC=', roc_auc_score(test_labels, test_scores), 'PR-AUC=', average_precision_score(test_labels, test_scores))

## 12) Интерпретация conditions

Можно посмотреть, как распределяются веса `w` по условиям для конкретной пары.

In [ ]:
@torch.no_grad()
def inspect_pair_weights(model: SCENet, processor: CLIPProcessor, img_path_1: str, img_path_2: str, image_cache: dict = None):
    model.eval()

    p1, p2 = str(img_path_1), str(img_path_2)
    if image_cache is not None:
        if p1 not in image_cache or p2 not in image_cache:
            raise KeyError(f'Path not in cache: {p1}, {p2}')
        img1, img2 = image_cache[p1], image_cache[p2]
    else:
        img1_bgr = cv2.imread(p1, cv2.IMREAD_COLOR)
        img2_bgr = cv2.imread(p2, cv2.IMREAD_COLOR)
        if img1_bgr is None or img2_bgr is None:
            raise FileNotFoundError(f'Cannot read one of images: {img_path_1}, {img_path_2}')
        img1 = cv2.cvtColor(img1_bgr, cv2.COLOR_BGR2RGB)
        img2 = cv2.cvtColor(img2_bgr, cv2.COLOR_BGR2RGB)

    px1 = processor(images=[img1], return_tensors='pt')['pixel_values'].to(device)
    px2 = processor(images=[img2], return_tensors='pt')['pixel_values'].to(device)

    _, _, w, _, _ = model.forward_pair(px1, px2)
    w = w[0].cpu().numpy()
    for i, wi in enumerate(w, 1):
        print(f'C{i}: {wi:.4f}')
    return w

# sample = test_df.iloc[0]
# inspect_pair_weights(model, processor, sample['sku1_path'], sample['sku2_path'], image_cache=IMAGE_CACHE)

## FAQ по коду (ответы на ваши вопросы)

### Что делает `def make_triplet_collate`?
Это фабрика `collate_fn` для `DataLoader`: она собирает список триплетов в батч и прогоняет изображения через `CLIPProcessor`, чтобы модель получила тензоры `pixel_values`.

### Для чего в `TripletImageDataset` передается `self.processor = processor`?
Для явной конфигурации dataset и для будущего расширения пайплайна (аугментации/процессинг в `__getitem__`). В текущем варианте основная обработка в `collate_fn`.

### Раскрой каждую строчку в классе `SCENet`
Сделано комментариями прямо внутри класса: инициализация backbone, вычисление D, masks `C`, weight-branch `w`, Eq.1/2/3 и `forward_pair`.

### Что делает `encode_image(...)`?
Извлекает визуальный вектор и L2-нормализует его. Нормализация стабилизирует масштаб эмбеддингов и расстояния для metric-learning.

### Что происходит в `sce_loss(...)`?
Считаем `d_pos`/`d_neg`, triplet hinge (Eq.4), добавляем L1/L2 (Eq.5), суммируем в итоговый objective.

### Почему `d_pos = ||Ea-Ep||`, `d_neg = ||Ea-En||`?
Потому что это суть triplet metric learning: positive ближе к anchor, negative дальше.

### Что происходит в `run_epoch`?
Один проход по всем батчам: forward для `(a,p)` и `(a,n)`, loss, шаг оптимизатора (в train), сбор метрик. Дополнительно добавлен proxy ROC-AUC на триплет-парах.


### Почему заменили PIL на OpenCV?
`cv2.imread` обычно быстрее декодирует JPEG/PNG в batch-пайплайнах и часто снижает `data_time_s` при обучении на GPU.

## 13) Оптимизация throughput на A100: как подбирать параметры

Ниже практический чек-лист, если GPU-util низкий и "прыгает":

1. **Смотрите ratio `data_time_s / compute_time_s`** (мы добавили его в логи):
   - если `data_time_s` сопоставим или больше `compute_time_s` → bottleneck на input pipeline.
2. **Подбор `num_workers`**:
   - начните с `[4, 8, 12, 16]` и выберите, где минимальный `data_time_s`.
   - слишком большие значения тоже вредят (oversubscription CPU/IO).
3. **`prefetch_factor`**:
   - типично 2-8. Если батчи тяжёлые по decode/resize, увеличивайте.
4. **`persistent_workers=True`**:
   - важно, чтобы worker-ы не перезапускались каждую эпоху.
5. **AMP (`use_amp=True`)**:
   - на A100 обычно заметный прирост скорости и снижение памяти.
6. **Anchor encode один раз**:
   - `forward_triplet` устраняет дублирующийся прогон backbone.
7. **Если `freeze_backbone=True` и данных мало**:
   - можно предвычислить image embeddings `V` один раз и обучать только head (ещё быстрее).
8. **Мониторинг**:
   - параллельно держите `nvidia-smi dmon` или `watch -n 0.5 nvidia-smi`;
   - ориентируйтесь не только на пик util, но на время эпохи и стабильность.

### Почему последующие эпохи могли замедляться
- перезапуск DataLoader workers без persistent mode,
- накапливающаяся нагрузка на файловую систему при большом числе open/decode,
- неэффективный pipeline CPU→GPU (без non_blocking + pin_memory).


## 14) Что дальше улучшать

1. Тюнинг `M in [1,2,5,10]` по валидации (как в абляциях статьи).
2. Добавить semi-hard/hard negative mining.
3. Для production кэшировать image embeddings `V` офлайн и онлайн считать только weight-branch + маскирование.

Реализация выше следует статье по ключевым формулам Eq.1-Eq.5, но адаптирована к вашим парным меткам через построение триплетов.

## 15) Улучшения: loss-функции, интерпретация score и инференс через сохранённые эмбеддинги

### 15.1 Можно ли улучшить loss?
Да. Текущий `triplet hinge` — хороший базовый вариант, но в SCE-Net можно безопасно попробовать:

1. **Soft-margin triplet** (`softplus(d_pos - d_neg)`):
   - более гладкие градиенты,
   - часто стабильнее при шумной разметке.
2. **Cosine-margin ranking**:
   - если используем L2-нормированные эмбеддинги, cosine-геометрия обычно интерпретируемее.
3. **Circle loss**:
   - можно использовать, но лучше в режиме "anchor + много positive/negative в batch".
   - для строго одного `(a,p,n)` на шаг выгода обычно меньше, чем от мягкого triplet/cosine-margin.

Ниже добавлены drop-in функции `sce_loss_soft_triplet` и `sce_loss_cosine_margin`.

### 15.2 Что означает score `-||e1 - e2||_2`?
Это **монотонная мера близости**:
- чем меньше расстояние между эмбеддингами, тем score выше (менее отрицательный, ближе к 0);
- чем пары дальше, тем score ниже (более отрицательный).

Поэтому ваши медианы `Good > Normal > Bad` выглядят логично:
- `Good`: ближе, score выше;
- `Bad`: дальше, score ниже.

Важно: абсолютные значения (например `-0.56`) сами по себе не "вероятность".
Обычно score калибруют на val-наборе (AUC/PR-AUC + подбор threshold).

### 15.3 Стоит ли считать cosine на инференсе?
Да, это нормальный и часто более интерпретируемый вариант:
- cosine in [-1, 1], где выше = похожее;
- особенно полезно, если эмбеддинги нормированы (`F.normalize`).

Ниже добавлены режимы `metric='neg_l2' | 'cosine'` и конвертация cosine в [0,1] через `(cos+1)/2`.

### 15.4 Можно ли сначала сохранить эмбеддинги, а потом считать score?
Да, и это правильно для production:
1. прогоняем каждое изображение **один раз** → сохраняем `path -> embedding`;
2. для любых пар потом быстро считаем score без повторного forward.

Важный нюанс: в SCE-Net финальный `E` зависит от пары через веса conditions `w(a,b)`.
Поэтому есть 2 режима:
- **pair-aware (точный)**: через `forward_pair` для каждой пары (дороже, но максимально соответствует обучению);
- **single-image embedding (быстрый)**: через `encode_image` и сравнение эмбеддингов (быстрее и кэшируемо).

Рекомендуемый компромисс: offline retrieval/filtering на cached cosine, rerank топ-k pair-aware score.


In [ ]:
# --- Альтернативные лоссы (drop-in вместо sce_loss) ---
def sce_loss_soft_triplet(e_anchor, e_pos, e_neg, condition_masks, lambda_l1=1e-6, lambda_l2=1e-6):
    d_pos = torch.norm(e_anchor - e_pos, p=2, dim=-1)
    d_neg = torch.norm(e_anchor - e_neg, p=2, dim=-1)

    # Smooth triplet: softplus(d_pos - d_neg)
    triplet = F.softplus(d_pos - d_neg).mean()
    l1 = condition_masks.abs().mean()
    l2 = (condition_masks ** 2).mean()
    return triplet + lambda_l1 * l1 + lambda_l2 * l2, triplet.item(), l1.item(), l2.item()


def sce_loss_cosine_margin(e_anchor, e_pos, e_neg, condition_masks, margin=0.2, lambda_l1=1e-6, lambda_l2=1e-6):
    # Чем выше cosine, тем более похожи
    s_pos = F.cosine_similarity(e_anchor, e_pos, dim=-1)
    s_neg = F.cosine_similarity(e_anchor, e_neg, dim=-1)

    # Хотим s_pos >= s_neg + margin
    ranking = F.relu(margin - s_pos + s_neg).mean()
    l1 = condition_masks.abs().mean()
    l2 = (condition_masks ** 2).mean()
    return ranking + lambda_l1 * l1 + lambda_l2 * l2, ranking.item(), l1.item(), l2.item()


@torch.no_grad()
def build_embedding_cache(model: SCENet, image_paths, processor: CLIPProcessor, batch_size: int = 128, image_cache: dict = None):
    """Кэш path -> single-image embedding (encode_image), чтобы не гонять модель повторно."""
    model.eval()
    emb_cache = {}
    paths = list(map(str, sorted(set(image_paths))))

    for i in tqdm(range(0, len(paths), batch_size), desc='Building embedding cache'):
        chunk = paths[i:i+batch_size]
        imgs = []
        for p in chunk:
            if image_cache is not None:
                if p not in image_cache:
                    raise KeyError(f'Path not found in image_cache: {p}')
                imgs.append(image_cache[p])
            else:
                bgr = cv2.imread(p, cv2.IMREAD_COLOR)
                if bgr is None:
                    raise FileNotFoundError(f'Cannot read image: {p}')
                imgs.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

        px = processor(images=imgs, return_tensors='pt')['pixel_values'].to(device)
        e = model.encode_image(px)  # [B, D], L2-normalized
        e = e.detach().cpu().numpy()
        for p, vec in zip(chunk, e):
            emb_cache[p] = vec

    return emb_cache


@torch.no_grad()
def pair_scores_pair_aware(model: SCENet, pair_df: pd.DataFrame, processor: CLIPProcessor, batch_size: int = 64,
                           image_cache: dict = None, metric: str = 'neg_l2'):
    """
    Pair-aware скоринг через forward_pair (учитывает w(a,b) в SCE-Net).
    metric: 'neg_l2' или 'cosine'
    """
    model.eval()
    scores = []
    rows = pair_df[['sku1_path', 'sku2_path']].astype(str).values.tolist()

    for i in tqdm(range(0, len(rows), batch_size), desc='Pair-aware scoring'):
        chunk = rows[i:i+batch_size]
        imgs1, imgs2 = [], []
        for a, b in chunk:
            if image_cache is not None:
                if a not in image_cache or b not in image_cache:
                    raise KeyError(f'Pair path not found in cache: {a}, {b}')
                imgs1.append(image_cache[a]); imgs2.append(image_cache[b])
            else:
                a_bgr = cv2.imread(a, cv2.IMREAD_COLOR); b_bgr = cv2.imread(b, cv2.IMREAD_COLOR)
                if a_bgr is None or b_bgr is None:
                    raise FileNotFoundError(f'Cannot read image in pair: {a}, {b}')
                imgs1.append(cv2.cvtColor(a_bgr, cv2.COLOR_BGR2RGB))
                imgs2.append(cv2.cvtColor(b_bgr, cv2.COLOR_BGR2RGB))

        px1 = processor(images=imgs1, return_tensors='pt')['pixel_values'].to(device)
        px2 = processor(images=imgs2, return_tensors='pt')['pixel_values'].to(device)
        e1, e2, _, _, _ = model.forward_pair(px1, px2)

        if metric == 'neg_l2':
            sc = -torch.norm(e1 - e2, p=2, dim=-1)
        elif metric == 'cosine':
            sc = F.cosine_similarity(e1, e2, dim=-1)
        else:
            raise ValueError(f'Unknown metric: {metric}')
        scores.extend(sc.cpu().numpy().tolist())

    return np.array(scores)


def pair_scores_from_embedding_cache(pair_df: pd.DataFrame, emb_cache: dict, metric: str = 'cosine'):
    """
    Быстрый оффлайн скоринг пар из заранее сохранённых single-image embeddings.
    """
    scores = []
    for a, b in pair_df[['sku1_path', 'sku2_path']].astype(str).values.tolist():
        if a not in emb_cache or b not in emb_cache:
            raise KeyError(f'Pair path not found in emb_cache: {a}, {b}')
        v1 = torch.from_numpy(emb_cache[a])
        v2 = torch.from_numpy(emb_cache[b])

        if metric == 'cosine':
            s = F.cosine_similarity(v1[None], v2[None], dim=-1).item()  # [-1, 1]
        elif metric == 'neg_l2':
            s = (-torch.norm(v1 - v2, p=2)).item()
        else:
            raise ValueError(f'Unknown metric: {metric}')
        scores.append(s)

    return np.array(scores)


# Пример использования:
# all_paths = pd.concat([train_df['sku1_path'], train_df['sku2_path'], val_df['sku1_path'], val_df['sku2_path'], test_df['sku1_path'], test_df['sku2_path']]).astype(str).unique()
# EMB_CACHE = build_embedding_cache(model, all_paths, processor, batch_size=256, image_cache=IMAGE_CACHE)
# test_scores_cos = pair_scores_from_embedding_cache(test_df, EMB_CACHE, metric='cosine')
# print('TEST cosine cache ROC-AUC:', roc_auc_score(test_df['y'].values, test_scores_cos))


## 16) Независимые эмбеддинги на базе SCE-Net (для retrieval/scoring без парного forward)

Вы правильно заметили ограничение текущего SCE-Net:
- в `forward_pair(a,b)` итоговые эмбеддинги зависят от парных весов `w(a,b)`;
- значит, "финальный" pair-aware embedding нельзя честно закешировать по одному айтему и потом сравнивать как независимые векторы.

Но архитектуру можно переделать так, чтобы получить **independent embeddings** и сохранить идею SCE:

### Вариант A: Global SCE embedding (condition pooling)
1. Для одного изображения считаем `V`.
2. Применяем condition masks `C_j ⊙ V`.
3. Вместо парного `w(a,b)` используем **single-image pooling** `α(x)`:
   - либо глобальные обучаемые веса `α` (одни для всех),
   - либо image-conditioned MLP `V -> α(x)`.
4. Получаем `E_global(x) = Σ_j α_j(x) * (C_j ⊙ V)`.
5. На инференсе: cosine(`E_global(item1)`, `E_global(item2)`).

Плюсы: полностью независимый оффлайн-кэш; минусы: теряем pair-specific адаптацию.

### Вариант B: Multi-head embeddings
- Из `E_global` строим несколько head-эмбеддингов (`K` штук),
- обучаем triplet-loss по каждому head (или weighted sum),
- при инференсе считаем `mean cosine` по head-ам.

Это помогает захватывать разные "оси совместимости" (цвет, стиль, сезон и т.д.).

### Вариант C: Two-stage (практический компромисс)
1. Stage-1: independent `E_global` для ANN retrieval (быстро).
2. Stage-2: pair-aware `forward_pair` для rerank top-k (точно).

Обычно это лучший производственный режим: дешёвый recall + точный rerank.


In [ ]:
# --- Independent SCE variant: global embedding + optional multi-head ---
class SCENetGlobal(nn.Module):
    """
    SCE-вариант с независимыми эмбеддингами:
    - нет pair-wise weight branch w(a,b),
    - есть condition pooling alpha(x) для одного изображения,
    - можно добавить multi-head embeddings.
    """
    def __init__(self, clip_model: CLIPModel, num_conditions: int = 5, num_heads: int = 1,
                 head_dim: int = 256, image_conditioned_pooling: bool = True):
        super().__init__()
        self.clip = clip_model
        self.M = num_conditions
        self.K = num_heads
        self.image_conditioned_pooling = image_conditioned_pooling

        # Размерность image features
        self.D = self.clip.config.projection_dim

        # SCE masks C_j in R^D
        self.condition_masks = nn.Parameter(torch.randn(self.M, self.D) * 0.02)

        # alpha(x): global or image-conditioned
        if self.image_conditioned_pooling:
            self.pool_mlp = nn.Sequential(
                nn.Linear(self.D, self.D),
                nn.ReLU(),
                nn.Linear(self.D, self.M)
            )
        else:
            self.global_alpha_logits = nn.Parameter(torch.zeros(self.M))

        # Multi-head projections from E_global
        self.heads = nn.ModuleList([nn.Linear(self.D, head_dim) for _ in range(self.K)])

    def encode_image(self, pixel_values: torch.Tensor):
        v = self.clip.get_image_features(pixel_values=pixel_values)
        v = F.normalize(v, dim=-1)
        return v

    def global_sce_embedding(self, pixel_values: torch.Tensor):
        """Возвращает E_global и все head-эмбеддинги (L2-нормированные)."""
        v = self.encode_image(pixel_values)  # [B, D]

        # O_j = C_j ⊙ V
        O = self.condition_masks.unsqueeze(0) * v.unsqueeze(1)  # [B, M, D]

        # alpha(x)
        if self.image_conditioned_pooling:
            alpha = torch.softmax(self.pool_mlp(v), dim=-1)  # [B, M]
        else:
            alpha = torch.softmax(self.global_alpha_logits, dim=-1).unsqueeze(0).expand(v.size(0), -1)

        # E_global = sum_j alpha_j O_j
        e_global = torch.sum(alpha.unsqueeze(-1) * O, dim=1)  # [B, D]
        e_global = F.normalize(e_global, dim=-1)

        head_embs = [F.normalize(head(e_global), dim=-1) for head in self.heads]  # K x [B, H]
        return e_global, head_embs, alpha


def multihead_triplet_loss(a_heads, p_heads, n_heads, margin: float = 0.2):
    """Triplet-loss усредняется по всем head-ам."""
    losses = []
    for ah, ph, nh in zip(a_heads, p_heads, n_heads):
        d_pos = torch.norm(ah - ph, p=2, dim=-1)
        d_neg = torch.norm(ah - nh, p=2, dim=-1)
        losses.append(F.relu(d_pos - d_neg + margin).mean())
    return torch.stack(losses).mean()


@torch.no_grad()
def build_global_embedding_cache(model: SCENetGlobal, image_paths, processor: CLIPProcessor,
                                 batch_size: int = 128, image_cache: dict = None):
    """Кэш независимых SCE-эмбеддингов: path -> {'global': vec, 'heads': [..]}"""
    model.eval()
    out = {}
    paths = list(map(str, sorted(set(image_paths))))

    for i in tqdm(range(0, len(paths), batch_size), desc='Caching global SCE embeddings'):
        chunk = paths[i:i+batch_size]
        imgs = []
        for p in chunk:
            if image_cache is not None:
                imgs.append(image_cache[p])
            else:
                bgr = cv2.imread(p, cv2.IMREAD_COLOR)
                if bgr is None:
                    raise FileNotFoundError(f'Cannot read image: {p}')
                imgs.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

        px = processor(images=imgs, return_tensors='pt')['pixel_values'].to(device)
        e_global, heads, _ = model.global_sce_embedding(px)

        e_global = e_global.cpu().numpy()
        heads = [h.cpu().numpy() for h in heads]
        for bi, p in enumerate(chunk):
            out[p] = {
                'global': e_global[bi],
                'heads': [h[bi] for h in heads]
            }
    return out


def score_pair_from_global_cache(a_path: str, b_path: str, emb_cache: dict, mode: str = 'mean_head_cosine'):
    """
    mode:
      - 'global_cosine': cosine(E_global(a), E_global(b))
      - 'mean_head_cosine': средний cosine по всем head-ам
    """
    va = emb_cache[str(a_path)]
    vb = emb_cache[str(b_path)]

    if mode == 'global_cosine':
        x = torch.from_numpy(va['global'])[None]
        y = torch.from_numpy(vb['global'])[None]
        return F.cosine_similarity(x, y, dim=-1).item()

    if mode == 'mean_head_cosine':
        sims = []
        for ha, hb in zip(va['heads'], vb['heads']):
            x = torch.from_numpy(ha)[None]
            y = torch.from_numpy(hb)[None]
            sims.append(F.cosine_similarity(x, y, dim=-1).item())
        return float(np.mean(sims))

    raise ValueError(f'Unknown mode: {mode}')


# Итого:
# - Да, можно обучить на triplet и получить независимые эмбеддинги.
# - Да, тогда на инференсе футболка/джинсы сравниваются напрямую cosine.
# - Цена вопроса: чуть ниже точность vs pair-aware SCE, поэтому часто делают two-stage pipeline.


## 17) Triplet-training + independent embeddings через compatibility memory (attention)

Да, это можно сделать именно так, как вы описали:
- **обучаемся на триплетах** (чтобы выучить паттерны совместимости),
- на инференсе получаем **независимый embedding каждого айтема**,
- потом считаем cosine между двумя item-эмбеддингами.

Ключевая идея: добавить в модель **память compatibility-прототипов** (learnable memory bank) и attention к ним.
Тогда каждый item кодируется как "базовый visual embedding + смесь глобальных паттернов совместимости".

### Архитектура (Compatibility Memory Encoder)
Для каждого изображения `x`:
1. `v(x)` — базовый visual embedding (CLIP).
2. `q(x)=W_q v(x)` — query.
3. `M in R^{P x D}` — `P` обучаемых compatibility-прототипов (memory slots).
4. `a(x)=softmax(q(x) M^T / sqrt(D))` — attention-веса по прототипам.
5. `c(x)=a(x) M` — compatibility context.
6. `z(x)=normalize(W_o [v(x), c(x)])` — итоговый независимый embedding.

Именно `z(x)` кэшируется на инференсе по каждому item отдельно.

### Как обучать, чтобы "запомнить паттерны"
Используем комбинированный objective:
- `L_triplet(z_a, z_p, z_n)` — основной сигнал совместимости,
- `L_distill` (опционально): студент `z` приближается к pair-aware teacher score,
- `L_diversity` по памяти: чтобы прототипы не схлопывались в один.

Итог: `L = L_triplet + λ_distill * L_distill + λ_div * L_diversity`.

Это даёт "латентное многомерное пространство совместимости" для каждого item, которое затем можно использовать в retrieval/ANN.


In [ ]:
class CompatibilityMemoryEncoder(nn.Module):
    """
    Независимый item-encoder с attention к compatibility memory.
    Обучается на триплетах, инференс — по одному айтему.
    """
    def __init__(self, clip_model: CLIPModel, emb_dim: int = 512, num_prototypes: int = 32):
        super().__init__()
        self.clip = clip_model
        self.D = self.clip.config.projection_dim
        self.P = num_prototypes

        # Learnable compatibility memory (global patterns)
        self.memory = nn.Parameter(torch.randn(self.P, self.D) * 0.02)

        # Query / Output projections
        self.q_proj = nn.Linear(self.D, self.D)
        self.out_proj = nn.Linear(self.D * 2, emb_dim)

    def encode_visual(self, pixel_values: torch.Tensor):
        v = self.clip.get_image_features(pixel_values=pixel_values)
        return F.normalize(v, dim=-1)

    def encode_item(self, pixel_values: torch.Tensor):
        """Возвращает независимый item embedding z и attention к прототипам."""
        v = self.encode_visual(pixel_values)              # [B, D]
        q = self.q_proj(v)                                # [B, D]

        attn_logits = (q @ self.memory.t()) / (self.D ** 0.5)  # [B, P]
        attn = torch.softmax(attn_logits, dim=-1)         # [B, P]

        ctx = attn @ self.memory                          # [B, D]
        z = self.out_proj(torch.cat([v, ctx], dim=-1))    # [B, emb_dim]
        z = F.normalize(z, dim=-1)
        return z, attn


def memory_diversity_loss(memory: torch.Tensor):
    """Штрафуем корреляцию прототипов между собой (кроме диагонали)."""
    m = F.normalize(memory, dim=-1)
    sim = m @ m.t()  # [P, P]
    eye = torch.eye(sim.size(0), device=sim.device)
    return ((sim - eye) ** 2).mean()


def triplet_with_memory_loss(model: CompatibilityMemoryEncoder,
                             a_px: torch.Tensor, p_px: torch.Tensor, n_px: torch.Tensor,
                             margin: float = 0.2, lambda_div: float = 1e-3):
    z_a, _ = model.encode_item(a_px)
    z_p, _ = model.encode_item(p_px)
    z_n, _ = model.encode_item(n_px)

    d_pos = torch.norm(z_a - z_p, p=2, dim=-1)
    d_neg = torch.norm(z_a - z_n, p=2, dim=-1)
    l_triplet = F.relu(d_pos - d_neg + margin).mean()

    l_div = memory_diversity_loss(model.memory)
    loss = l_triplet + lambda_div * l_div
    return loss, {'triplet': l_triplet.item(), 'diversity': l_div.item()}


@torch.no_grad()
def build_item_embedding_cache(model: CompatibilityMemoryEncoder, image_paths, processor: CLIPProcessor,
                               batch_size: int = 128, image_cache: dict = None):
    """Инференс-кэш: path -> независимый embedding z(x)."""
    model.eval()
    out = {}
    paths = list(map(str, sorted(set(image_paths))))

    for i in tqdm(range(0, len(paths), batch_size), desc='Caching item embeddings'):
        chunk = paths[i:i+batch_size]
        imgs = []
        for p in chunk:
            if image_cache is not None:
                if p not in image_cache:
                    raise KeyError(f'Path not found in cache: {p}')
                imgs.append(image_cache[p])
            else:
                bgr = cv2.imread(p, cv2.IMREAD_COLOR)
                if bgr is None:
                    raise FileNotFoundError(f'Cannot read image: {p}')
                imgs.append(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))

        px = processor(images=imgs, return_tensors='pt')['pixel_values'].to(device)
        z, _ = model.encode_item(px)
        z = z.cpu().numpy()
        for p, zi in zip(chunk, z):
            out[p] = zi

    return out


def score_pairs_from_item_cache(pair_df: pd.DataFrame, item_emb_cache: dict):
    """Скоринг пар через cosine по независимым z(x)."""
    scores = []
    for a, b in pair_df[['sku1_path', 'sku2_path']].astype(str).values.tolist():
        if a not in item_emb_cache or b not in item_emb_cache:
            raise KeyError(f'Path missing in item cache: {a}, {b}')
        x = torch.from_numpy(item_emb_cache[a])[None]
        y = torch.from_numpy(item_emb_cache[b])[None]
        scores.append(F.cosine_similarity(x, y, dim=-1).item())
    return np.array(scores)


# Практика:
# 1) train на triplets -> model learns global compatibility prototypes in memory
# 2) cache z(x) for all catalog items
# 3) ANN / cosine retrieval by z(x)
# 4) (опционально) rerank top-k pair-aware SCE score
